# Preprocess

In [ ]:
# The following code is a preprocess for data being used to train document intellgence model.
# The code does as follows:
# 1. Reads PDF files from a specified directory.
# 2. Converts each page of the PDF into an image format (JPEG).
# 3. Saves the images in a specified output directory.
# 4. Optionally, it can also convert the images to grayscale for better OCR performance


In [ ]:
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pdf2image import convert_from_path

def convert_pdfs_to_images(folder_path, output_folder, dpi=300, poppler_path=None):
    os.makedirs(output_folder, exist_ok=True)
    failed_folder = os.path.join(folder_path, "failed_files")

    for filename in os.listdir(folder_path):
        # 1. SKIP UNWANTED FILES (.DS_Store, hidden files, etc.)
        if filename.startswith('.') or filename.startswith('~$'):
            continue
            
        file_path = os.path.join(folder_path, filename)

        # Process PDFs
        if filename.lower().endswith('.pdf'):
            try:
                images = convert_from_path(file_path, dpi=dpi, poppler_path=poppler_path)

                for i, page in enumerate(images):
                    # Convert image to OpenCV format (BGR then Grayscale)
                    open_cv_image = np.array(page)
                    # Convert RGB to BGR for OpenCV
                    open_cv_image = cv2.cvtColor(open_cv_image, cv2.COLOR_RGB2BGR)
                    # GRAYSCALE 
                    grey_image = cv2.cvtColor(open_cv_image, cv2.COLOR_BGR2GRAY)
                    
                    image_filename = f"{os.path.splitext(filename)[0]}_page_{i+1}.jpg"
                    save_path = os.path.join(output_folder, image_filename)
                    cv2.imwrite(save_path, grey_image)

                print(f"✅ Converted & Grayscaled PDF: {filename}")

            except Exception as e:
                print(f"❌ FAILED to convert {filename}: {e}")
                os.makedirs(failed_folder, exist_ok=True)
                shutil.move(file_path, os.path.join(failed_folder, filename))

        # Process existing Images
        elif filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img = cv2.imread(file_path)
            if img is not None:
                grey_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
                cv2.imwrite(os.path.join(output_folder, filename), grey_img)
                print(f"📸 Processed existing image to Gray: {filename}")

def remove_small_artifacts(input_folder, output_folder, min_area=40):
    os.makedirs(output_folder, exist_ok=True)
    for file in os.listdir(input_folder):
        if file.startswith('.'): continue
        
        image_path = os.path.join(input_folder, file)
        grey = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if grey is None: continue
        
        # Adaptive threshold to isolate text from noisy background
        binary = cv2.adaptiveThreshold(grey, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                       cv2.THRESH_BINARY_INV, 11, 8)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
        mask = np.zeros_like(binary)
        
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= min_area:
                mask[labels == i] = 255

        clean_img = cv2.bitwise_not(mask)
        cv2.imwrite(os.path.join(output_folder, f"cleaned_{file}"), clean_img)



In [ ]:
# --- EXECUTION ---

# Paths
folder_path = '/Users/hanamengistu/Documents/Practicum1/practicum1.0/FinalProjectCode/Data/dataset3'
image_folder = '/Users/hanamengistu/Documents/Practicum1/practicum1.0/FinalProjectCode/Pre/Preprocess/pdf_to_image/'
cleaned_output_folder = "/Users/hanamengistu/Documents/Practicum1/practicum1.0/FinalProjectCode/Pre/Preprocess/removed_small_artifacts/"

# Step 1: PDF to Grayscale Image
convert_pdfs_to_images(folder_path, image_folder, dpi=300, poppler_path="/opt/homebrew/bin")



In [4]:
# Step 2: Artifact Removal (Clean the noise)
remove_small_artifacts(image_folder, cleaned_output_folder)

print("🚀 All files processed, grayscaled, and cleaned!")

Premature end of JPEG file


KeyboardInterrupt: 